## Ensemble_Agent Experimenting

### Imports

In [4]:
from deal_hunter.agents.items import Item

In [5]:
#sys
from pathlib import Path
import sys
root_dir = Path.cwd().resolve().parents[0]
sys.path.insert(0,str(root_dir/"src"))
sys.path.insert(0,str(root_dir/"notebooks"))

In [6]:
#imports
import os 
from dotenv import load_dotenv
from huggingface_hub import login
from tqdm.auto import tqdm



### Load Dataset

In [7]:
#hf login
load_dotenv(override = True)
hf_token = os.environ["HF_TOKEN"]
login(token=hf_token , add_to_git_credential = False)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [8]:
pricer_data = "Vishy08/pricer-data"
train, test_items, val = Item.from_hub(dataset_name=pricer_data)
print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test_items):,} test items")

Loaded 320,000 training items, 4,000 validation items, 8,000 test items


In [9]:
test_items[0]

Item(title='ZLINE 30 in. Wooden Wall Mount Range Hood in Cottage White - Includes Motor (KPTT-30)', price=899.95, category='Appliances', test_prompt="How much does this cost to the nearest dollar?\n\nZLINE 30 in. Wooden Wall Mount Range Hood in Cottage White - Includes Motor\nThe ZLINE KPTT wall mount wooden range hood designed to be both elegant and powerful, featuring the industry's only lifetime warranty motor. The hand-finished painted cottage white wood is made from solid pine with a stainless steel inner frame, making it both durable and long-lasting. This durable construction, along with ZLINE's lifetime warranty on the motor, guarantees a range hood that will last for a lifetime. This classic wooden hood contains many modern features, such as Hand-carved, hand-finished wood; Dishwasher-safe stainless steel baffle filters; Built-in LED lighting; High performance motor with speeds up to 400 CFM. All ZLINE range hoods come equipped with everything needed to easily install and use,

### Testing ChromaDB

In [10]:
import chromadb
DB = str(root_dir/"notebooks"/"products_vectorstore")
chroma_client = chromadb.PersistentClient(path = DB)

### Encoder LLM (all-Mini-L6-v2)

In [11]:
#Using SentenceTransformer Encoding LLM ( Free(fast) + Private)
from sentence_transformers import SentenceTransformer
encoder_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7046.49it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
sentences = [
    item.test_prompt for item in test_items[:50]
]
embeddings = encoder_model.encode(sentences=sentences)


In [13]:
type(embeddings)

numpy.ndarray

### Building ChromaDB 

In [14]:
def summarizer(item):
    question = "How much does this cost to the nearest dollar?\n\n"
    prefix = "\n\nPrice is $"
    summary = item.test_prompt.replace(question,"").replace(prefix,"")
    return f"{summary}"

In [15]:
summarizer(test_items[3000])

'POWERTEC 70107V 4 T-Fitting for Dust Collection Hose, 1 PK\nIntroducing the 4-Inch T-Fitting by POWERTEC. This handy t-shaped connector optimizes the performance of your dust collection system by allowing you to add branches (connections) to your setup without compromising its efficiency. This creates a secure, airtight connection to ensure that harmful dust, dirt and debris is sucked in and expelled directly into your dust collector – resulting in clean air and a clean work environment. This dust collector hose adapter is ideal for professionals, hobbyists and workshops where space constraints or machine design are a concern. T-Shaped Fitting Dimensions 4-Inch OD nominal ends measure O.D. and I.D. Premium Build The POWERTEC T-Fitting is made out of high quality ABS plastic. This lightweight material is commonly known for it’s super durability'

In [16]:
def build_product_collection(chroma_client , encoder_model, items, name="products", batch_size=1000):
    
    collection = chroma_client.get_or_create_collection(name) 
    #avoiding creation if already exist
    try :
        if collection.count()>0 :
            return collection
    except Exception :
        existing =  collection.get(limit = 1 , include = [])
        if existing.get("ids"):
            return collection    

    # processing data in batch of 1000 
    with tqdm(total = len(items) , desc=f"Chroma {name}:", unit = "doc",dynamic_ncols = True) as pbar:
        for start in range(0,len(items),batch_size):
            #creating the batch
            batch = items[start : start + batch_size] 
            
            #include title + description 
            docs = [summarizer(item) for item in batch]
            embeddings = encoder_model.encode(docs).astype(float).tolist()

            metas = [{"category" : item.category , "price" : item.price} for item in batch]
            doc_ids = [f"{name}_{start + k}" for k in range(len(batch))]

            collection.upsert(
                documents = docs,
                metadatas =metas,
                ids = doc_ids,
                embeddings = embeddings
            )
            pbar.update(len(batch))
    
    return collection


In [17]:
collection = build_product_collection(chroma_client, encoder_model, train[:])

### Running and Similarity Search

In [18]:
chroma_client.get_collection("products")


Collection(name=products)

In [19]:
text = ["POWERTEC 70107V 4 T-Fitting for Dust Collection Hose, 1 PK\nIntroducing the 4-Inch T-Fitting by POWERTEC. This handy t-shaped connector optimizes the performance of your dust collection system by allowing you to add branches (connections) to your setup without compromising its efficiency. This creates a secure, airtight connection to ensure that harmful dust, dirt and debris is sucked in and expelled directly into your dust collector – resulting in clean air and a clean work environment. This dust collector hose adapter is ideal for professionals, hobbyists and workshops where space constraints or machine design are a concern. T-Shaped Fitting Dimensions 4-Inch OD nominal ends measure O.D. and I.D. Premium Build The POWERTEC T-Fitting is made out of high quality ABS plastic. This lightweight material is commonly known for it’s super durability"]
result = collection.query( query_texts=text,n_results = 3)
print(result["documents"][0][:])

['TERABITHIA Mini 10inch Realistic Couple Girls Children Birthday Xmas Gift Reborn Baby Doll Silicone Vinyl Full Body Newborn Twins\nCute and realistic reborn doll full body are made of high quality silicone-like vinyl.Safe non-toxic,pure environmentally friendly materials. Can pose a lot like sitting and lying,can not speaking. Full reborn doll body can enter into the water. The doll will come with outfits just like the picture. It will be a growth partners of your baby and the one of your family. Wish you love and take care of her (him). *These dolls are not toys;they are fine collectibles to be enjoyed by adult collectors. Attention The default is the baby girl, if you need baby boy, please contact us via email directly. You will get baby doll *2; 2.Doll clothes(May be some differences between each batch);']


In [20]:
#creating a function to find similar and convert the given text to embedding

def find_similar(text):
    #converting text to embedding
    vector = encoder_model.encode(text) 
    #finding n similar based on embedding generated using text
    result = collection.query(
        query_embeddings=vector ,
        include = ["documents","metadatas"],
        n_results=5,
        )
    document = result["documents"][0][:]
    price = [m["price"] for m in result["metadatas"][0][:]]
    return document , price

In [21]:
find_similar(text)

(['TERABITHIA Mini 10inch Realistic Couple Girls Children Birthday Xmas Gift Reborn Baby Doll Silicone Vinyl Full Body Newborn Twins\nCute and realistic reborn doll full body are made of high quality silicone-like vinyl.Safe non-toxic,pure environmentally friendly materials. Can pose a lot like sitting and lying,can not speaking. Full reborn doll body can enter into the water. The doll will come with outfits just like the picture. It will be a growth partners of your baby and the one of your family. Wish you love and take care of her (him). *These dolls are not toys;they are fine collectibles to be enjoyed by adult collectors. Attention The default is the baby girl, if you need baby boy, please contact us via email directly. You will get baby doll *2; 2.Doll clothes(May be some differences between each batch);'],
 [49.99])

###  Creating context for LLM 

In [22]:
def make_context(similars, prices):
    message = "Here are the similar products with prices, which can be helpful for context\n"
    for similar , price in zip(similars, prices):
        message += f"Related Product:\n{similar}\nPrice is ${price}\n\n"
    return message

In [23]:
similars , prices = find_similar(text)
context = make_context(similars,prices)
print(context)

Here are the similar products with prices, which can be helpful for context
Related Product:
TERABITHIA Mini 10inch Realistic Couple Girls Children Birthday Xmas Gift Reborn Baby Doll Silicone Vinyl Full Body Newborn Twins
Cute and realistic reborn doll full body are made of high quality silicone-like vinyl.Safe non-toxic,pure environmentally friendly materials. Can pose a lot like sitting and lying,can not speaking. Full reborn doll body can enter into the water. The doll will come with outfits just like the picture. It will be a growth partners of your baby and the one of your family. Wish you love and take care of her (him). *These dolls are not toys;they are fine collectibles to be enjoyed by adult collectors. Attention The default is the baby girl, if you need baby boy, please contact us via email directly. You will get baby doll *2; 2.Doll clothes(May be some differences between each batch);
Price is $49.99




In [24]:
#appending context in GPT message format
#Using item to test the modal performance
def message_for(item,similars,prices):
    message = f"Estimate the price of the product. Respond with Price only No Explanation\n\n{summarizer(item)}"
    message += make_context(similars,prices)
    return [{"role":"user" ,"content":message}]

In [25]:
similars , prices = find_similar(summarizer(train[2345]))
print(message_for(train[2345] , similars,prices))

[{'role': 'user', 'content': "Estimate the price of the product. Respond with Price only No Explanation\n\nMoen CSI Arris Modern Hand -Towel Ring, 6.50 x 3.15 x 5.71 inches, Brushed Nickel\nProduct Description Arris faucets and accessories feature a cylindrical look that's thoroughly modern. Sharp angles add distinctive contrast to the tubular lines that dominate each piece in this collection. Exposed pipes and industrial-chic finishes are just a few of the downtown loft-inspired design details that come to life in Arris faucets and accessories. Sharp angles and tubular lines dominate each piece in this modern collection. Amazon.com Modern style with Moen functionality ( view larger ). Exposed pipes and industrial-chic finishes are just a few of the downtown loft-inspired design details that come to life in Moen's Arris faucets and accessories. Sharp angles and tubular lines dominate each piece in this stylish and functional modern collection. For a coordinated look toHere are the simi

### Checking Response and Testing

In [26]:
from litellm import completion
import os 
from dotenv import load_dotenv
load_dotenv(override=True)

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]


In [27]:
def gpt_5_1_rag(item):
    documents, prices = find_similar(summarizer(item))
    response = completion(model="gpt-5.1",messages=message_for(item,documents,prices),reasoning_effort="none",seed=42)
    return response.choices[0].message.content

In [28]:
response = gpt_5_1_rag(train[56678])
print(response)

$189.99


In [29]:
import importlib
import deal_hunter.services.testing as testing

importlib.reload(testing)
# Do not ``import test`` from this module — it shadows the HF ``test`` split name.
from deal_hunter.services.testing import Tester, evaluate

### Testing


In [30]:
import importlib
import re

import pandas as pd

import deal_hunter.services.testing as testing

importlib.reload(testing)
from deal_hunter.agents.items import Item
from deal_hunter.services.testing import Tester, evaluate


def _parse_price(text: object) -> float:
    s = str(text).replace("$", "").replace(",", "")
    m = re.search(r"[-+]?\d*\.?\d+", s)
    return float(m.group()) if m else 0.0


def gpt_5_1_rag_row(row) -> float:
    item = Item.model_validate(row.to_dict())
    raw = gpt_5_1_rag(item)
    return _parse_price(raw)


test_df = pd.DataFrame([x.model_dump() for x in test_items]).reset_index(drop=True)
n = min(250, len(test_df))

evaluate(gpt_5_1_rag_row, test_df, title="GPT-5.1 RAG", size=n)
# or: Tester(gpt_5_1_rag_row, test_df, title="GPT-5.1 RAG", size=n).run()

Correct Predictions: 100%|██████████| 175/1754:55<00:00]
Testing Progress: 100%|██████████| 250/250 [04:55<00:00]



Test Results Summary:
Total Predictions: 250
Correct (Green): 188 (75.2%)
Close (Orange): 42 (16.8%)
Wrong (Red): 20 (8.0%)
Average Error: $38.02
RMSLE: 0.51
